<a href="https://colab.research.google.com/github/parinyad123/financial-analyst-agent/blob/main/notebooks/financial_analyst_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Financial Analyst Agent — Colab Dev Notebook

**Physics-informed financial analysis** ที่ผสม quantitative signals (Hurst exponent) กับ LLM reasoning ผ่าน ReAct agent

| Component | Technology |
|---|---|
| LLM (dev) | Groq — `openai/gpt-oss-120b` (ประหยัด Gemini quota) |
| Agent | LangGraph `create_react_agent` (ReAct pattern) |
| Market data | yfinance |
| Observability | LangSmith — project: `financial-analyst-agent` |

**Flow การทำงาน:**
```
User query → ReAct Agent (gpt-oss-120b)
                ├── get_stock_price      → yfinance
                ├── get_stock_financials → yfinance
                └── get_hurst_exponent   → yfinance + numpy (R/S analysis)
                        ↓
             LangSmith (trace ทุก step)
```

> ⚠️ **ก่อนรัน:** ตั้งค่า Colab Secrets (🔑 ไอคอนซ้ายมือ): `LANGCHAIN_API_KEY`, `GROQ_API_KEY` และเปิด Notebook access

**กฎการรัน:** รันจากบนลงล่างเท่านั้น — ถ้า Cell 2 (Verify) ไม่ผ่าน ห้ามรันต่อ

---
## Cell 1 — Install dependencies + Imports

ติดตั้งทุก package ในที่เดียว แล้ว import ทั้งหมด — รวมไว้ cell เดียวเพื่อไม่ให้เกิดปัญหา import order
(บทเรียนจากรอบ debug: langsmith cache ค่า env ตอน import ครั้งแรก ดังนั้นเราเลิกพึ่ง env แล้วใช้ explicit binding แทน → ดู Cell 2)

In [1]:
# ============================================================
# Cell 1: Install + Imports
# ============================================================
!pip install -q -U langsmith langchain-groq langgraph yfinance langchain-core

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from datetime import datetime

import numpy as np
import yfinance as yf
import json
import pandas as pd
import logging
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

from google.colab import userdata

# LangChain / LangGraph
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.tracers import LangChainTracer
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent

# LangSmith
import langsmith
from langsmith import Client, traceable, tracing_context
from langsmith.run_helpers import get_current_run_tree

print("✅ All imports OK")
print("langsmith version:", langsmith.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 498.0/498.0 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.8/137.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.2 MB/s eta 0:00:00
✅ All imports OK
langsmith version: 0.8.15


---
## Cell 2 — LangSmith client (explicit binding)

**ทำไมไม่ใช้ env vars (`LANGCHAIN_TRACING_V2` ฯลฯ)?**
`langsmith.utils.get_env_var` ถูก cache ด้วย `lru_cache` — ถ้า import ก่อน set env ค่าจะค้างเป็น "disabled" ตลอด session ใน notebook ที่ cell order ไม่แน่นอน นี่คือระเบิดเวลา

**แนวทางที่ใช้:** bind `api_key` + `project_name` ตรง ๆ เข้า `Client`, `LangChainTracer`, และ `@traceable` ทุกจุด — ไม่พึ่ง env เลย
(ตอน deploy จริงใน Docker/FastAPI ค่อยกลับไปใช้ env ได้ เพราะ env ถูก set ตอน container start ก่อน import เสมอ)

**Assertion ท้าย cell:** ถ้า print ❌ ห้ามรัน cell ถัดไป — เช็ค API key ใน Colab Secrets ก่อน

In [2]:
# ============================================================
# Cell 2: LangSmith client + tracer (explicit binding — ไม่พึ่ง env)
# ============================================================
PROJECT_NAME = "financial-analyst-agent"

ls_client = Client(
    api_key=userdata.get("LANGCHAIN_API_KEY"),
    api_url="https://api.smith.langchain.com",
)

# tracer สำหรับ LangGraph agent — ผูก project + client ตรง ๆ
tracer = LangChainTracer(
    project_name=PROJECT_NAME,
    client=ls_client,
)

# สร้าง project ถ้ายังไม่มี + ใช้เป็น connectivity check ไปในตัว
try:
    project = ls_client.read_project(project_name=PROJECT_NAME)
    print(f"✅ Project exists: {project.name}")
except Exception:
    project = ls_client.create_project(
        PROJECT_NAME,
        description="Financial Analyst Agent with ReAct + Hurst Exponent",
    )
    print(f"✅ Project created: {project.name}")

# 🛑 Gate: ต้องผ่านก่อนรันต่อ
assert project is not None, "❌ เชื่อม LangSmith ไม่ได้ — เช็ค LANGCHAIN_API_KEY ใน Colab Secrets"
print("✅ LangSmith ready — ไปต่อได้")

✅ Project exists: financial-analyst-agent
✅ LangSmith ready — ไปต่อได้


---
## Cell 3 — Tools: price, financials, Hurst exponent

**Pattern สำคัญ — `@tool` นอกสุด, `@traceable` ห่อ logic ข้างใน:**

```python
@tool                      # ← LLM เห็น docstring นี้ ใช้ตัดสินใจเลือก tool
def tool_name(x):
    return _logic(x)

@traceable(run_type="tool", client=ls_client)   # ← LangSmith trace ตัวนี้
def _logic(x): ...
```

แยกกันเพราะ: `@tool` ทำหน้าที่ interface กับ LLM (schema + docstring) ส่วน `@traceable` ทำหน้าที่ observability — ถ้อยซ้อน decorator บนฟังก์ชันเดียวกันจะตีกัน

| Tool | ข้อมูลที่คืน | Tags ใน LangSmith |
|---|---|---|
| `get_stock_price` | ราคา, % change, 52W range, P/E, market cap | `market-data`, `yfinance` |
| `get_stock_financials` | revenue, net income, margin, growth, EPS, D/E | `fundamentals`, `yfinance` |
| `get_hurst_exponent` | Hurst exponent (R/S analysis) + market regime | `quant`, `regime-detection` |

**Hurst exponent อ่านยังไง:** H > 0.55 → Trending (momentum ใช้ได้) | H < 0.45 → Mean-Reverting (RSI/Bollinger) | ระหว่างนั้น → Random Walk

In [3]:
# ============================================================
# Cell 3: Tools — get_stock_price / get_stock_financials / get_hurst_exponent
# ============================================================

# ---------- Tool 1: ราคาปัจจุบัน + key metrics ----------
@tool
def get_stock_price(ticker: str) -> str:
    """Fetch current stock price and key metrics."""
    return _fetch_stock_price_logic(ticker)

@traceable(
    name="fetch_stock_price",
    run_type="tool",
    tags=["market-data", "yfinance"],
    client=ls_client,
)
def _fetch_stock_price_logic(ticker: str) -> str:
    try:
        stock = yf.Ticker(ticker.upper())
        hist = stock.history(period="5d")   # 5d เผื่อวันที่ market ปิด
        if hist.empty:
            return f"No data for {ticker}"

        latest = hist['Close'].iloc[-1]
        prev = hist['Close'].iloc[-2] if len(hist) > 1 else latest
        change_pct = ((latest - prev) / prev) * 100
        info = stock.info

        pos_in_range = (latest - info.get('fiftyTwoWeekLow', latest)) / \
               (info.get('fiftyTwoWeekHigh', latest) - info.get('fiftyTwoWeekLow', latest) + 1e-9) * 100

        return (
            f"Ticker: {ticker.upper()}\n"
            f"Price: ${latest:.2f} (Change: {change_pct:+.2f}%)\n"
            f"52W Range: ${info.get('fiftyTwoWeekLow','N/A')} – ${info.get('fiftyTwoWeekHigh','N/A')}\n"
            f"P/E (TTM): {info.get('trailingPE','N/A')} | Forward P/E: {info.get('forwardPE','N/A')}\n"
            f"Market Cap: ${info.get('marketCap',0)/1e9:.1f}B\n"
            f"Position in 52W range: {pos_in_range:.0f}%"
        )
    except Exception as e:
        return f"Error: {str(e)}"


# ---------- Tool 2: Fundamentals ----------
@tool
def get_stock_financials(ticker: str) -> str:
    """Get fundamental financial metrics for analysis."""
    return _fetch_financials_logic(ticker)

@traceable(
    name="fetch_financials",
    run_type="tool",
    tags=["fundamentals", "yfinance"],
    client=ls_client,
)
def _fetch_financials_logic(ticker: str) -> str:
    try:
        info = yf.Ticker(ticker.upper()).info
        return (
            f"Revenue (TTM): ${info.get('totalRevenue',0)/1e9:.1f}B\n"
            f"Net Income: ${info.get('netIncomeToCommon',0)/1e9:.1f}B\n"
            f"Profit Margin: {info.get('profitMargins',0)*100:.1f}%\n"
            f"Revenue Growth YoY: {info.get('revenueGrowth',0)*100:.1f}%\n"
            f"EPS (TTM): ${info.get('trailingEps','N/A')}\n"
            f"Debt/Equity: {info.get('debtToEquity','N/A')}"
        )
    except Exception as e:
        return f"Error: {str(e)}"


# ---------- Tool 3: Hurst exponent (R/S analysis) ----------
@tool
def get_hurst_exponent(ticker: str) -> str:
    """Calculate Hurst exponent to detect market regime."""
    return _calc_hurst_logic(ticker)

@traceable(
    name="calc_hurst_exponent",
    run_type="tool",
    tags=["quant", "regime-detection"],
    client=ls_client,
)
def _calc_hurst_logic(ticker: str) -> str:
    try:
        # 1) log returns จาก 1Y daily close
        hist = yf.Ticker(ticker.upper()).history(period="1y")["Close"]
        returns = np.log(hist / hist.shift(1)).dropna().values

        # 2) Rescaled Range (R/S) ต่อ lag — แบ่ง series เป็น segments
        lags = range(2, 20)
        rs_values = []
        for lag in lags:
            segments = [returns[i:i+lag] for i in range(0, len(returns)-lag, lag)]
            rs_list = [
                (np.max(np.cumsum(s - np.mean(s))) - np.min(np.cumsum(s - np.mean(s)))) / np.std(s)
                for s in segments if np.std(s) > 0
            ]
            if rs_list:
                rs_values.append(np.mean(rs_list))

        # 3) Hurst = slope ของ log(R/S) vs log(lag)
        hurst = np.polyfit(np.log(list(lags)[:len(rs_values)]), np.log(rs_values), 1)[0]

        # 4) จำแนก regime
        if hurst > 0.55:
            regime = "📈 Trending — momentum strategies work"
        elif hurst < 0.45:
            regime = "↔️ Mean-Reverting — RSI/Bollinger strategies work"
        else:
            regime = "🎲 Random Walk — harder to predict"

        return f"Hurst Exponent ({ticker.upper()}, 1Y): {hurst:.4f}\nRegime: {regime}"
    except Exception as e:
        return f"Error: {str(e)}"


print("✅ Tools ready:", [t.name for t in [get_stock_price, get_stock_financials, get_hurst_exponent]])

✅ Tools ready: ['get_stock_price', 'get_stock_financials', 'get_hurst_exponent']


---
## Cell 4–6 — Tools เพิ่มเติม (TODO ตาม build order)

- **Cell 4:** `analyze_portfolio_risk` (UC-2a) — Volatility, Sharpe, Sortino, VaR/CVaR 95%, Max Drawdown, correlation matrix
- **Cell 5:** `search_market_news` — Gemini 2.0 Flash + Google Search grounding (ใช้ model แยกจาก agent หลัก)
- **Cell 6:** `track_portfolio` (UC-2b) — dev ด้วย `MOCK_PORTFOLIO` dict ก่อน, swap เป็น PostgreSQL ตอน deploy

In [4]:
# ============================================================
# Cell 4: TODO — analyze_portfolio_risk (UC-2a)
# Input: {ticker: weight}, weights ต้องรวม 1.0 ± 0.01
# ============================================================

# Mock portfolio สำหรับทดสอบ UC-2b (Cell 6)
MOCK_PORTFOLIO = {
    "positions": [
        {"ticker": "NVDA", "shares": 10, "avg_cost": 150.0},
        {"ticker": "AMD",  "shares": 20, "avg_cost": 100.0},
        {"ticker": "TSLA", "shares":  5, "avg_cost": 200.0},
    ]
}

In [5]:
# ============================================================
# Cell 4: Tool — analyze_portfolio_risk (UC-2a)
# ============================================================

TRADING_DAYS = 252
RISK_FREE_RATE = 0.045      # ~3M T-Bill — v1 hardcode, v2 ค่อยดึง ^IRX จาก yfinance
MIN_HISTORY_DAYS = 60       # ขั้นต่ำที่ metrics พอเชื่อถือได้ (กัน ticker เพิ่ง IPO)


@tool
def analyze_portfolio_risk(portfolio: str) -> str:
    """USE THIS TOOL for any portfolio risk assessment question
    (e.g. "ประเมิน risk ของ portfolio", "วิเคราะห์ความเสี่ยงพอร์ต").
    Computes ALL risk metrics in one call: volatility, Sharpe, Sortino,
    VaR/CVaR 95%, max drawdown, correlation matrix.
    Do NOT call get_stock_price or get_stock_financials for portfolio risk —
    this tool fetches all required data itself.
    Input MUST be a JSON string: '{"NVDA": 0.5, "AMD": 0.3, "TSLA": 0.2}'.
    Weights must sum to 1.0."""
    return _portfolio_risk_logic(portfolio)


@traceable(
    name="portfolio_risk_analysis",
    run_type="tool",
    tags=["quant", "risk"],
    client=ls_client,
)
def _portfolio_risk_logic(portfolio) -> str:
    # ---------- 1) Parse + validate input ----------
    # เผื่อ model ส่ง dict มาตรง ๆ (บาง model ไม่ serialize เป็น string)
    if isinstance(portfolio, dict):
        weights_dict = portfolio
    else:
        try:
            weights_dict = json.loads(portfolio)
        except (json.JSONDecodeError, TypeError) as e:
            return f'Error: invalid input — ต้องเป็น JSON string เช่น {{"NVDA": 0.5, "AMD": 0.5}} ({e})'

    if not isinstance(weights_dict, dict) or not weights_dict:
        return "Error: portfolio ว่าง หรือ format ไม่ใช่ {ticker: weight}"

    try:
        weights_dict = {str(t).upper().strip(): float(w) for t, w in weights_dict.items()}
    except (ValueError, TypeError):
        return "Error: weight ทุกตัวต้องเป็นตัวเลข เช่น 0.5 ไม่ใช่ '50%'"

    if any(w <= 0 for w in weights_dict.values()):
        return "Error: weight ต้องเป็นบวกทุกตัว (v1 ยังไม่รองรับ short positions)"

    total = sum(weights_dict.values())
    if abs(total - 1.0) > 0.01:
        return (
            f"Error: weights รวมได้ {total:.3f} ต้องรวม 1.0 ± 0.01 — "
            f"ให้แจ้ง user ปรับ weights เอง (tool จะไม่ normalize ให้)"
        )

    tickers = sorted(weights_dict)   # yf.download คืน columns เรียง alphabet — sort ให้ตรงกันไว้ก่อน

    # ---------- 2) Download 1Y daily close ----------
    try:
        data = yf.download(tickers, period="1y", progress=False, auto_adjust=True)["Close"]
    except Exception as e:
        return f"Error: ดึงข้อมูลจาก yfinance ไม่สำเร็จ — {e}"

    # Edge: ticker เดียว → yfinance อาจคืน Series ไม่ใช่ DataFrame
    if isinstance(data, pd.Series):
        data = data.to_frame(name=tickers[0])

    # Edge: ticker พิมพ์ผิด / delisted → คอลัมน์เป็น NaN ทั้งแถบ
    dead = [t for t in tickers if t not in data.columns or data[t].isna().all()]
    if dead:
        return f"Error: ไม่พบข้อมูลของ {dead} — เช็ค ticker symbol อีกครั้ง"

    # Align dates: ตัดวันที่บาง ticker ไม่มีข้อมูล (IPO ใหม่ / ตลาดคนละประเทศ)
    data = data[tickers].dropna()
    if len(data) < MIN_HISTORY_DAYS:
        return (
            f"Error: ข้อมูลที่ทุก ticker มีร่วมกันมีแค่ {len(data)} วัน "
            f"(ขั้นต่ำ {MIN_HISTORY_DAYS}) — อาจมี ticker ที่เพิ่ง IPO"
        )

    # ---------- 3) Returns + portfolio series ----------
    returns = np.log(data / data.shift(1)).dropna()

    # ⚠️ จุดพังคลาสสิก: weights ต้อง align ตาม column order ของ DataFrame
    # ไม่ใช่ order ใน dict ที่ user ส่งมา
    weights = np.array([weights_dict[t] for t in data.columns])
    port_returns = returns.values @ weights

    # ---------- 4) Metrics (annualized, 252 trading days) ----------
    ann_return = port_returns.mean() * TRADING_DAYS
    ann_vol = port_returns.std(ddof=1) * np.sqrt(TRADING_DAYS)
    sharpe = (ann_return - RISK_FREE_RATE) / ann_vol if ann_vol > 0 else float("nan")

    # Edge: portfolio ที่แทบไม่มีวันติดลบ → downside std = 0 → division by zero
    downside = port_returns[port_returns < 0]
    if len(downside) > 1 and downside.std(ddof=1) > 0:
        sortino_str = f"{(ann_return - RISK_FREE_RATE) / (downside.std(ddof=1) * np.sqrt(TRADING_DAYS)):.2f}"
    else:
        sortino_str = "N/A (แทบไม่มีวันติดลบใน 1Y)"

    var_95 = np.percentile(port_returns, 5)
    cvar_95 = port_returns[port_returns <= var_95].mean()

    # Max Drawdown — port_returns เป็น log returns → ใช้ exp(cumsum) ตรงกว่า cumprod(1+r)
    cum = np.exp(np.cumsum(port_returns))
    peak = np.maximum.accumulate(cum)
    max_dd = ((cum - peak) / peak).min()

    # Per-ticker annualized volatility
    per_vol = returns.std(ddof=1) * np.sqrt(TRADING_DAYS)
    per_vol_str = "\n".join(f"  {t}: {v * 100:.1f}%" for t, v in per_vol.items())

    # Edge: correlation matrix ต้องมี >= 2 tickers
    if len(tickers) >= 2:
        corr_str = returns.corr().round(2).to_string()
    else:
        corr_str = "N/A (single asset — ไม่มี correlation)"

    return (
        f"Portfolio: {weights_dict}\n"
        f"Period: 1Y daily ({len(returns)} trading days)\n"
        f"Annualized Return: {ann_return * 100:+.1f}%\n"
        f"Annualized Volatility: {ann_vol * 100:.1f}%\n"
        f"Sharpe Ratio: {sharpe:.2f} (rf = {RISK_FREE_RATE * 100:.1f}%)\n"
        f"Sortino Ratio: {sortino_str}\n"
        f"VaR 95% (daily): {var_95 * 100:.2f}% | CVaR 95% (daily): {cvar_95 * 100:.2f}%\n"
        f"Max Drawdown: {max_dd * 100:.1f}%\n"
        f"Per-Ticker Volatility (annualized):\n{per_vol_str}\n"
        f"Correlation Matrix:\n{corr_str}"
    )


print("✅ analyze_portfolio_risk ready")

# ---------- Smoke test: เรียก logic ตรง ๆ ก่อนผ่าน agent ----------
# (เทสนอก agent ก่อน — แยกปัญหา "logic พัง" ออกจาก "model เรียก tool เพี้ยน")
print("\n--- Test 1: happy path ---")
print(_portfolio_risk_logic('{"NVDA": 0.5, "AMD": 0.3, "TSLA": 0.2}'))

print("\n--- Test 2: weights ไม่ครบ 1.0 ---")
print(_portfolio_risk_logic('{"NVDA": 0.5, "AMD": 0.3}'))

print("\n--- Test 3: ticker มั่ว ---")
print(_portfolio_risk_logic('{"ZZZFAKE123": 1.0}'))

print("\n--- Test 4: single ticker ---")
print(_portfolio_risk_logic('{"NVDA": 1.0}'))

print("\n--- Test 5: JSON พัง ---")
print(_portfolio_risk_logic("NVDA 50% AMD 50%"))

✅ analyze_portfolio_risk ready

--- Test 1: happy path ---
Portfolio: {'NVDA': 0.5, 'AMD': 0.3, 'TSLA': 0.2}
Period: 1Y daily (250 trading days)
Annualized Return: +64.8%
Annualized Volatility: 36.7%
Sharpe Ratio: 1.64 (rf = 4.5%)
Sortino Ratio: 2.33
VaR 95% (daily): -3.84% | CVaR 95% (daily): -5.17%
Max Drawdown: -22.5%
Per-Ticker Volatility (annualized):
  AMD: 65.5%
  NVDA: 35.1%
  TSLA: 44.6%
Correlation Matrix:
Ticker   AMD  NVDA  TSLA
Ticker                  
AMD     1.00  0.49  0.35
NVDA    0.49  1.00  0.37
TSLA    0.35  0.37  1.00

--- Test 2: weights ไม่ครบ 1.0 ---
Error: weights รวมได้ 0.800 ต้องรวม 1.0 ± 0.01 — ให้แจ้ง user ปรับ weights เอง (tool จะไม่ normalize ให้)

--- Test 3: ticker มั่ว ---
Error: ไม่พบข้อมูลของ ['ZZZFAKE123'] — เช็ค ticker symbol อีกครั้ง

--- Test 4: single ticker ---
Portfolio: {'NVDA': 1.0}
Period: 1Y daily (250 trading days)
Annualized Return: +35.0%
Annualized Volatility: 35.1%
Sharpe Ratio: 0.87 (rf = 4.5%)
Sortino Ratio: 1.34
VaR 95% (daily): -3

---
## Cell 7 — Agent setup (Groq + ReAct)

**ทำไม Groq ไม่ใช่ Gemini ตอน dev:** Gemini free tier ถอด quota ของ `gemini-2.0-flash` ออกแล้ว (`limit: 0` → 429 ตลอด) — dev ใน Colab ใช้ Groq แล้วค่อย swap เป็น Gemini ตอน deploy

**ทำไม `gpt-oss-120b` ไม่ใช่ Llama 3.3 70B:** reasoning model ที่ train มาเพื่อ agentic tasks → tool orchestration เสถียรกว่า + ถูกกว่า (`reasoning_effort="low"` พอสำหรับ tool routing และประหยัด output tokens)

⚠️ Trade-off: ภาษาไทยของ gpt-oss อ่อนกว่า Llama 3.3 (ที่รองรับไทย official) — ถ้า output ไทยเพี้ยน สลับ model ได้ด้วยการเปลี่ยน string เดียว

In [6]:
# ============================================================
# Cell 7: Agent setup — ChatGroq + create_react_agent
# ============================================================

# Dev: Groq | Production: swap เป็น ChatGoogleGenerativeAI(model="gemini-2.0-flash")
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    reasoning_effort="low",   # พอสำหรับ tool orchestration, ประหยัด tokens
    api_key=userdata.get("GROQ_API_KEY"),
)

tools = [get_stock_price, get_stock_financials, get_hurst_exponent, analyze_portfolio_risk]

SYSTEM_PROMPT = """You are a quantitative financial analyst assistant.
Always fetch real-time data before answering.
NEVER state any number that did not come from a tool result in this conversation.
If you lack data, call the appropriate tool or say you don't have it — do not estimate from memory.
For portfolio risk questions, use analyze_portfolio_risk.
Provide objective analysis with data. Note that this is not financial advice.
Do not give specific price targets, entry points, or stop-loss levels.
Respond in Thai mixed with English technical terms."""
agent_graph = create_react_agent(model, tools, prompt=SYSTEM_PROMPT)
print("✅ Agent graph ready")

✅ Agent graph ready


---
## Cell 8 — `run_financial_agent` (entry point + tracing wrapper)

หน้าที่ของ wrapper นี้:
1. **`@traceable`** — group ทุก sub-runs (LLM calls + tool calls) ไว้ใน 1 parent trace
2. **`callbacks=[tracer]`** — ส่ง LangGraph internal runs เข้า LangSmith แบบ explicit (ไม่พึ่ง env)
3. **`metadata` + `tags`** — filter ใน LangSmith UI ได้ตาม ticker / analysis_type
4. **`run_id` ใน return** — ดึงจาก `get_current_run_tree()` ภายใน function → ได้ ID ของ trace นี้เป๊ะ ๆ ไม่ต้อง query ย้อนหลัง (และคือ `trace_id` ที่ FastAPI endpoint ต้อง return ตาม spec)

In [7]:
# ============================================================
# Cell 8: run_financial_agent — main entry point
# ============================================================

@traceable(
    name="financial_analyst_agent",
    run_type="chain",
    tags=["agent", "financial-analysis"],
    project_name=PROJECT_NAME,
    client=ls_client,
)
def run_financial_agent(
    query: str,
    tickers: list[str] = None,
    analysis_type: str = "general",
) -> dict:
    """Main entry point — group ทุก sub-runs ไว้ใน 1 parent trace"""
    config = RunnableConfig(
        run_name=f"query_{analysis_type}_{datetime.now().strftime('%H%M%S')}",
        callbacks=[tracer],                       # explicit tracer — ไม่พึ่ง env
        tags=[analysis_type] + (tickers or []),
        metadata={
            "query": query,
            "tickers": tickers or [],
            "analysis_type": analysis_type,
            "timestamp": datetime.now().isoformat(),
        },
    )

    inputs = {"messages": [HumanMessage(content=query)]}
    final_response = ""

    print(f"\n{'='*55}")
    print(f"🔍 Query: {query[:80]}...")
    print(f"{'='*55}")

    # stream_mode="values" → ได้ state เต็มทุก step, print ทุก message (Human/AI/Tool)
    for event in agent_graph.stream(inputs, config=config, stream_mode="values"):
        if "messages" in event:
            last = event["messages"][-1]
            last.pretty_print()
            if hasattr(last, "content") and last.content:
                final_response = last.content

    # ดึง run ID ของ trace นี้จากข้างใน — แม่นกว่า list_runs ย้อนหลัง
    rt = get_current_run_tree()
    return {
        "query": query,
        "response": final_response,
        "tickers": tickers,
        "analysis_type": analysis_type,
        "run_id": str(rt.id) if rt else None,
    }

print("✅ run_financial_agent ready")

✅ run_financial_agent ready


---
## Cell 9 — Test UC-1: วิเคราะห์หุ้นรายตัว

ลำดับการทำงาน:
1. **`tracing_context(enabled=True, client=ls_client)`** — เปิด tracing ให้ `@traceable` ทุกตัวในก้อนนี้ (จำเป็นเพราะเราไม่ได้ set env)
2. **`ls_client.flush()`** — บังคับส่ง pending traces ทันที (ปกติส่งแบบ background batch)
3. **Trace URLs** — private URL (เปิดดูเองใน workspace) + public URL จาก `share_run()` (แปะใน README ให้คนอื่นดูได้โดยไม่ต้อง login)

In [8]:
# ============================================================
# Cell 9: Test UC-1 — single stock analysis
# ============================================================

with tracing_context(enabled=True, client=ls_client):
    result = run_financial_agent(
        query="วิเคราะห์ NVDA ให้หน่อย: ราคาปัจจุบัน, fundamentals, และ market regime (Hurst)",
        tickers=["NVDA"],
        analysis_type="full_analysis",
    )

ls_client.flush()   # บังคับส่ง traces ก่อน query หา run

# ---------- Trace URLs ----------
import time
time.sleep(5)       # เผื่อ server-side ingest

run_id = result["run_id"]
run = ls_client.read_run(run_id)
print(f"\n🔒 Private URL: {run.url}")

# Public URL — uncomment ถ้าต้องการ share (เช่นแปะใน README)
# shared_url = ls_client.share_run(run_id)
# print(f"🌐 Public URL: {shared_url}")


🔍 Query: วิเคราะห์ NVDA ให้หน่อย: ราคาปัจจุบัน, fundamentals, และ market regime (Hurst)...
================================ Human Message =================================

วิเคราะห์ NVDA ให้หน่อย: ราคาปัจจุบัน, fundamentals, และ market regime (Hurst)
================================== Ai Message ==================================
Tool Calls:
  get_stock_price (fc_8c522c6d-73d7-44dc-a55f-d429f26cde52)
 Call ID: fc_8c522c6d-73d7-44dc-a55f-d429f26cde52
  Args:
    ticker: NVDA
================================= Tool Message =================================
Name: get_stock_price

Ticker: NVDA
Price: $204.87 (Change: +2.22%)
52W Range: $140.85 – $236.54
P/E (TTM): 31.325687 | Forward P/E: 16.097084
Market Cap: $4962.2B
Position in 52W range: 67%
================================== Ai Message ==================================
Tool Calls:
  get_stock_financials (fc_1bf1f137-186d-4e89-9fb8-552497be475e)
 Call ID: fc_1bf1f137-186d-4e89-9fb8-552497be475e
  Args:
    ticker: NVDA
==============

---
## Cell 10–11 — Test UC-2a / UC-2b (TODO)

รอ tools จาก Cell 4–6 ก่อน:
- **UC-2a:** `run_financial_agent(query="ประเมิน risk ของ portfolio NVDA 50% AMD 30% TSLA 20%", analysis_type="portfolio_risk")`
- **UC-2b:** track P&L จาก `MOCK_PORTFOLIO`

In [10]:
# ============================================================
# Cell 10: Test UC-2a — portfolio risk analysis
# ============================================================
with tracing_context(enabled=True, client=ls_client):
    result = run_financial_agent(
        query="ประเมิน risk ของ portfolio NVDA 50% AMD 30% TSLA 20%",
        tickers=["NVDA", "AMD", "TSLA"],
        analysis_type="portfolio_risk",
    )
ls_client.flush()


🔍 Query: ประเมิน risk ของ portfolio NVDA 50% AMD 30% TSLA 20%...
================================ Human Message =================================

ประเมิน risk ของ portfolio NVDA 50% AMD 30% TSLA 20%
================================== Ai Message ==================================
Tool Calls:
  analyze_portfolio_risk (fc_8cb02738-02c2-4351-a784-f4886bbe7133)
 Call ID: fc_8cb02738-02c2-4351-a784-f4886bbe7133
  Args:
    portfolio: {"NVDA": 0.5, "AMD": 0.3, "TSLA": 0.2}
================================= Tool Message =================================
Name: analyze_portfolio_risk

Portfolio: {'NVDA': 0.5, 'AMD': 0.3, 'TSLA': 0.2}
Period: 1Y daily (250 trading days)
Annualized Return: +64.8%
Annualized Volatility: 36.7%
Sharpe Ratio: 1.64 (rf = 4.5%)
Sortino Ratio: 2.33
VaR 95% (daily): -3.84% | CVaR 95% (daily): -5.17%
Max Drawdown: -22.5%
Per-Ticker Volatility (annualized):
  AMD: 65.5%
  NVDA: 35.1%
  TSLA: 44.6%
Correlation Matrix:
Ticker   AMD  NVDA  TSLA
Ticker                  
AMD  

---
## Utility — ดู runs ย้อนหลังในโปรเจกต์

`list_runs(is_root=True)` เหมาะกับ sanity check / ไล่ดู history หลาย runs — ต่างจาก `result["run_id"]` ที่ได้ run ของ query นั้นเป๊ะ ๆ

In [11]:
# ============================================================
# Utility: list recent root runs
# ============================================================
runs = list(ls_client.list_runs(
    project_name=PROJECT_NAME,
    is_root=True,
    limit=5,
))
for r in runs:
    print(r.start_time, "|", r.name, "|", r.url)

2026-06-12 09:26:01.218137+00:00 | financial_analyst_agent | https://smith.langchain.com/o/ff247f3b-cff8-4723-8fdb-e22409b5199f/projects/p/b089e658-f958-4049-8dc5-a0bdaf661974/r/019ebb27-1502-79f2-abcb-b90091d0cb7e?trace_id=019ebb27-1502-79f2-abcb-b90091d0cb7e&start_time=2026-06-12T09:26:01.218137
2026-06-12 09:25:47.220266+00:00 | financial_analyst_agent | https://smith.langchain.com/o/ff247f3b-cff8-4723-8fdb-e22409b5199f/projects/p/b089e658-f958-4049-8dc5-a0bdaf661974/r/019ebb26-de54-78e3-b687-94a9843e1ccd?trace_id=019ebb26-de54-78e3-b687-94a9843e1ccd&start_time=2026-06-12T09:25:47.220266
2026-06-11 19:51:51.366561+00:00 | financial_analyst_agent | https://smith.langchain.com/o/ff247f3b-cff8-4723-8fdb-e22409b5199f/projects/p/b089e658-f958-4049-8dc5-a0bdaf661974/r/019eb83d-b146-74c3-97b8-3db90fb25e81?trace_id=019eb83d-b146-74c3-97b8-3db90fb25e81&start_time=2026-06-11T19:51:51.366561
2026-06-11 19:51:37.764627+00:00 | financial_analyst_agent | https://smith.langchain.com/o/ff247f3b-cff